# Práctica 2: Análisis Exploratorio de Datos (EDA)
## Actividad 6: Tratamiento de Desbalance - Telco Customer Churn

En este notebook se audita el desbalance de la variable objetivo de abandono (`Churn`), caracterizada por un desbalance moderado o alto asociado a la pérdida de clientes, aplicando las metodologías correspondientes[cite: 1, 3].

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Configuración de estilo
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans'})

carpeta_salida = "Desbalance Telco"
os.makedirs(carpeta_salida, exist_ok=True)

# 1. Cargar el dataset desde la carpeta 'datasets/'
ruta_telco = os.path.join("datasets", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_telco = pd.read_csv(ruta_telco)

df_telco['TotalCharges'] = pd.to_numeric(df_telco['TotalCharges'].astype(str).str.strip(), errors='coerce')
df_telco['TotalCharges'].fillna(df_telco['TotalCharges'].median(), inplace=True)
df_telco['tenure'].fillna(df_telco['tenure'].median(), inplace=True)
df_telco['MonthlyCharges'].fillna(df_telco['MonthlyCharges'].median(), inplace=True)

df_telco['Target'] = df_telco['Churn'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)
df_telco['Abandono_ES'] = df_telco['Target'].map({0: 'Permanece (No)', 1: 'Abandona (Sí)'})

cols_num_telco = ['tenure', 'MonthlyCharges', 'TotalCharges']
X = df_telco[cols_num_telco]
y = df_telco['Target']

### 1. Análisis de Frecuencias y Ratio de Desbalance (Telco)
Evaluamos la distribución de clientes que abandonan frente a los que permanecen mediante el cálculo del $IR$ para determinar la necesidad de intervención[cite: 3].

In [ ]:
conteo = y.value_counts()
porcentajes = y.value_counts(normalize=True) * 100

print("--- Conteo de observaciones por clase ---")
print(conteo)
print("\n--- Porcentaje por clase ---")
print(porcentajes)

mayoritaria = conteo.idxmax()
minoritaria = conteo.idxmin()
IR = conteo[mayoritaria] / conteo[minoritaria]
print(f"\nRazón de desbalance (IR): {IR:.2f}")

if IR > 1.5:
    print("-> El dataset presenta desbalance (IR > 1.5)[cite: 3].")

# Generar y guardar gráfica de distribución
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_telco, x='Abandono_ES', palette=['#2b5c8f', '#d95f02'])
plt.title("1. Distribución de Clases (Telco Churn)", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Estado del Cliente")
plt.ylabel("Frecuencia")

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())} ({p.get_height()/len(df_telco)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(carpeta_salida, "1_distribucion_clases_telco.png"), dpi=300)
plt.close()

### 2. Comparativa de Estrategias de Tratamiento
Aplicamos las técnicas de modelado contemplando la línea base, penalización de clases, remuestreo aleatorio y técnicas sintéticas (SMOTE), midiendo su impacto en la clasificación[cite: 3].

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_over, y_train_over = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)
X_train_under, y_train_under = RandomUnderSampler(random_state=42).fit_resample(X_train, y_train)
X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)

estrategias = {
    "2. Sin balanceo": (X_train, y_train, None),
    "3. Ponderación de clases": (X_train, y_train, "balanced"),
    "4. Sobremuestreo": (X_train_over, y_train_over, None),
    "5. Submuestreo": (X_train_under, y_train_under, None),
    "6. SMOTE": (X_train_smote, y_train_smote, None)
}

for idx, (nombre, (X_tr, y_tr, cw)) in enumerate(estrategias.items(), start=2):
    modelo = LogisticRegression(class_weight=cw, random_state=42, max_iter=1000)
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_test)
    
    print(f"\n================ ESTRATEGIA: {nombre} ================")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False)
    plt.title(f"Resultado - {nombre}", fontsize=12, pad=10, fontweight='bold')
    plt.xlabel("Predicción")
    plt.ylabel("Valor Real")
    plt.tight_layout()
    
    nombre_archivo = f"{idx}_matriz_resultado_{nombre.lower().replace(' ', '_').replace('.', '')}.png"
    plt.savefig(os.path.join(carpeta_salida, nombre_archivo), dpi=300)
    plt.close()

print(f"\n¡Proceso finalizado! Imágenes guardadas en '{carpeta_salida}'.")